# 03 — Güncel Stage 3 branch denetimi (no inference)

**Bu notebook 3 Kasım 2026 tarihli güncel P0/P1 deney kodunu ve 97-ID ek maruziyet filtresini doğrular.** 27 Ekim tarihsel v2 paketini kontrol eden `02_...` notebook'undan ayrıdır.

## Kaggle hazırlığı
1. GitHub PR #157 içindeki `research/2026-11-03-kaggle-single-pass-validation` branch'inin ZIP arşivini indir.
2. Arşivi aç; repository kökünü bir Kaggle Dataset olarak yükle ve bu notebook'a Input olarak ekle. Dataset içinde `artifacts/KAGGLE_STAGE3_CURRENT_BRANCH_AUDIT_2026-11-03.manifest.json` dosyası korunmalıdır.
3. GPU ve internet gerekmez. Notebook paket kurmaz/güncellemez, model indirmez/yüklemez, model yanıtı üretmez ve eğitim yapmaz.
4. Hücreleri sırayla çalıştır. Sonuç `/kaggle/working/stage3_current_branch_audit_*.zip` olacaktır.

## Beklenen sonuç
Kod/test denetimi geçerse `CODE_AUDIT_PASSED_RESEARCH_GATE_BLOCKED` raporlanmalıdır. Araştırma kapısının `ready=false` olması bu aşamada beklenen durumdur; vaka kabulü veya inference yetkisi vermez. Kaynak hash'i uyuşmazsa veya testler başarısızsa rapor denetimi başarısız olarak işaretlenir.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import importlib.metadata
import json
import platform
import subprocess
import sys
import zipfile

EXPECTED_MANIFEST_GIT_BLOB_SHA1 = 'fd8de87b36646a826adb09d958dff35f208d6be6'
EXPECTED_SOURCE_COMMIT = '429c9ed1ad410a1dcd2d395a885f9ac71f42de13'
EXPECTED_FILE_COUNT = 19
RUN_TS = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
WORK_ROOT = Path('/kaggle/working')
OUT_DIR = WORK_ROOT / f'stage3_current_branch_audit_{RUN_TS}'
OUT_DIR.mkdir(parents=True, exist_ok=False)
REPORT_PATH = OUT_DIR / 'audit_report.json'
HASH_LOG = OUT_DIR / 'source_integrity.json'
TEST_LOG = OUT_DIR / 'pytest.log'
PREFLIGHT_LOG = OUT_DIR / 'preflight.log'

def sha256_bytes(raw):
    return hashlib.sha256(raw).hexdigest()

def git_blob_sha1(raw):
    header = b'blob ' + str(len(raw)).encode('ascii') + b'\\0'
    return hashlib.sha1(header + raw).hexdigest()

def package_version(name):
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return None

report = {
    'audit_type': 'current_stage3_source_tests_preflight_no_inference',
    'expected_source_commit': EXPECTED_SOURCE_COMMIT,
    'runtime': {'python': platform.python_version(), 'platform': platform.platform(), 'pytest': package_version('pytest')},
    'source_integrity': {'status': 'not_started', 'expected_files': EXPECTED_FILE_COUNT, 'matched_files': 0, 'blockers': [], 'details': []},
    'tests': {'status': 'not_started', 'exit_code': None, 'test_files': []},
    'research_gate': {'status': 'not_started', 'ready': False, 'accepted_case_count': None, 'accepted_project_count': None, 'blockers': [], 'model_inference_performed': False},
    'model_loaded': False, 'model_inference_performed': False, 'model_training_performed': False,
}
manifest_name = 'KAGGLE_STAGE3_CURRENT_BRANCH_AUDIT_2026-11-03.manifest.json'
hits = list(Path('/kaggle/input').rglob(manifest_name))
if len(hits) != 1:
    report['source_integrity']['status'] = 'blocked_manifest_missing_or_not_unique'
    report['source_integrity']['blockers'].append(f'Expected exactly one {manifest_name}; found {len(hits)}.')
    REPORT_PATH.write_text(json.dumps(report, ensure_ascii=False, indent=2) + '\\n', encoding='utf-8')
    raise RuntimeError(report['source_integrity']['blockers'][-1])

manifest_path = hits[0]
manifest_raw = manifest_path.read_bytes()
actual_manifest_blob = git_blob_sha1(manifest_raw)
if actual_manifest_blob != EXPECTED_MANIFEST_GIT_BLOB_SHA1:
    report['source_integrity']['status'] = 'failed_manifest_hash_mismatch'
    report['source_integrity']['blockers'].append('Current-branch manifest Git blob SHA-1 mismatch.')
    REPORT_PATH.write_text(json.dumps(report, ensure_ascii=False, indent=2) + '\\n', encoding='utf-8')
    raise RuntimeError(report['source_integrity']['blockers'][-1])

manifest = json.loads(manifest_raw.decode('utf-8'))
repo_root = manifest_path.parent.parent
report['source_integrity']['manifest_path'] = str(manifest_path)
report['source_integrity']['manifest_sha256'] = sha256_bytes(manifest_raw)
report['source_integrity']['source_commit'] = manifest.get('source_commit')
entries = manifest.get('files')
if manifest.get('source_commit') != EXPECTED_SOURCE_COMMIT or manifest.get('model_execution_performed') is not False:
    report['source_integrity']['blockers'].append('Source commit or no-execution manifest field differs from the reviewed snapshot.')
if not isinstance(entries, list) or len(entries) != EXPECTED_FILE_COUNT:
    report['source_integrity']['blockers'].append('Current-branch manifest must contain exactly 19 source/test files.')
    entries = entries if isinstance(entries, list) else []
paths = [x.get('path') for x in entries if isinstance(x, dict)]
if len(paths) != len(entries) or len(paths) != len(set(paths)):
    report['source_integrity']['blockers'].append('Manifest file paths are malformed or duplicated.')
for entry in entries:
    rel = entry.get('path') if isinstance(entry, dict) else None
    expected_sha = entry.get('git_blob_sha1') if isinstance(entry, dict) else None
    expected_bytes = entry.get('byte_length') if isinstance(entry, dict) else None
    detail = {'path': rel, 'status': 'blocked'}
    if not isinstance(rel, str) or not rel or Path(rel).is_absolute() or '..' in Path(rel).parts:
        detail['error'] = 'Unsafe or invalid repository-relative path.'
        report['source_integrity']['blockers'].append(f'Invalid manifest path: {rel!r}')
        report['source_integrity']['details'].append(detail)
        continue
    target = repo_root / rel
    if not target.is_file():
        detail['error'] = 'File missing from uploaded repository snapshot.'
        report['source_integrity']['blockers'].append(f'Missing source file: {rel}')
    else:
        raw = target.read_bytes()
        actual_sha = git_blob_sha1(raw)
        detail.update({'actual_git_blob_sha1': actual_sha, 'expected_git_blob_sha1': expected_sha, 'actual_byte_length': len(raw), 'expected_byte_length': expected_bytes})
        if actual_sha == expected_sha and len(raw) == expected_bytes:
            detail['status'] = 'match'
            report['source_integrity']['matched_files'] += 1
        else:
            detail['error'] = 'Git blob SHA-1 or byte length mismatch.'
            report['source_integrity']['blockers'].append(f'Frozen current-branch source mismatch: {rel}')
    report['source_integrity']['details'].append(detail)
report['source_integrity']['status'] = 'passed' if not report['source_integrity']['blockers'] else 'failed'
HASH_LOG.write_text(json.dumps(report['source_integrity'], ensure_ascii=False, indent=2) + '\\n', encoding='utf-8')
print('Source integrity:', report['source_integrity']['status'], report['source_integrity']['matched_files'], '/', len(entries))
if report['source_integrity']['status'] != 'passed':
    report['overall_status'] = 'SOURCE_INTEGRITY_FAILED'
    REPORT_PATH.write_text(json.dumps(report, ensure_ascii=False, indent=2) + '\\n', encoding='utf-8')
    raise RuntimeError('Source integrity failed; tests and preflight are not run.')

test_files = [
    'tests/test_evidence_abstention_preflight.py',
    'tests/test_evidence_packet_audit.py',
    'tests/test_analyze_evidence_abstention_results.py',
    'tests/test_kaggle_stage3_prompt_contract.py',
    'tests/test_kaggle_no_inference_audit_notebook.py',
    'tests/test_build_evidence_abstention_cohort_matrix.py',
]
report['tests']['test_files'] = test_files
if package_version('pytest') is None:
    report['tests']['status'] = 'blocked_pytest_not_installed'
else:
    proc = subprocess.run([sys.executable, '-m', 'pytest', '-q', *test_files], cwd=str(repo_root), capture_output=True, text=True, errors='replace', timeout=900, check=False)
    test_output = (proc.stdout or '') + ('\\n--- STDERR ---\\n' + proc.stderr if proc.stderr else '')
    TEST_LOG.write_text(test_output, encoding='utf-8')
    report['tests']['exit_code'] = proc.returncode
    report['tests']['status'] = 'passed' if proc.returncode == 0 else 'failed'
print('Focused no-inference tests:', report['tests']['status'])
if TEST_LOG.exists():
    print(TEST_LOG.read_text(encoding='utf-8')[-10000:])

preflight_path = repo_root / 'scripts' / 'evidence_abstention_preflight.py'
if report['tests']['status'] == 'passed':
    pre = subprocess.run([sys.executable, str(preflight_path), '--root', str(repo_root)], cwd=str(repo_root), capture_output=True, text=True, errors='replace', timeout=120, check=False)
    PREFLIGHT_LOG.write_text((pre.stdout or '') + ('\\n--- STDERR ---\\n' + pre.stderr if pre.stderr else ''), encoding='utf-8')
    try:
        gate = json.loads(pre.stdout)
        report['research_gate'].update({'status': 'ready' if gate.get('ready') is True else 'blocked_as_expected', 'ready': gate.get('ready') is True, 'accepted_case_count': gate.get('accepted_case_count'), 'accepted_project_count': gate.get('accepted_project_count'), 'blocker_count': gate.get('blocker_count'), 'blockers': gate.get('blockers', []), 'model_inference_performed': gate.get('model_inference_performed', False), 'exit_code': pre.returncode})
    except json.JSONDecodeError:
        report['research_gate']['status'] = 'invalid_preflight_output'
        report['research_gate']['blockers'] = ['Preflight output was not valid JSON; inspect preflight.log.']
else:
    report['research_gate']['status'] = 'not_run_tests_failed'

if report['research_gate']['model_inference_performed'] is not False:
    raise RuntimeError('Unexpected model inference flag; audit must fail closed.')
report['model_loaded'] = False
report['model_inference_performed'] = False
report['model_training_performed'] = False
test_ok = report['tests']['status'] == 'passed'
gate_json_ok = report['research_gate']['status'] in {'ready', 'blocked_as_expected'}
if test_ok and gate_json_ok:
    report['overall_status'] = 'CODE_AUDIT_PASSED_RESEARCH_GATE_' + ('READY' if report['research_gate']['ready'] else 'BLOCKED')
else:
    report['overall_status'] = 'CURRENT_BRANCH_AUDIT_FAILED'
REPORT_PATH.write_text(json.dumps(report, ensure_ascii=False, indent=2) + '\\n', encoding='utf-8')
zip_path = WORK_ROOT / f'stage3_current_branch_audit_{RUN_TS}.zip'
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for item in sorted(OUT_DIR.rglob('*')):
        if item.is_file(): archive.write(item, arcname=str(item.relative_to(OUT_DIR)))
print('FINAL STATUS:', report['overall_status'])
print('Accepted cases:', report['research_gate']['accepted_case_count'])
print('Readiness blockers:', len(report['research_gate']['blockers']))
print('Report:', REPORT_PATH)
print('Bundle:', zip_path)
print('No model load, inference, or training was performed.')
